In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from collections import defaultdict

import matplotlib.pyplot as plt
import scipy.stats

def count_coverage_df(df, chrom):
    # Create an empty list for events
    events = []
    # Populate events with start (+1) and end (-1) markers
    for _, row in df.iterrows():
        events.append((row['bpStart'], 1))
        events.append((row['bpEnd'] + 1, -1))  # End is exclusive
    # Sort events by position
    events.sort()
    # Compute coverage using a sweep line algorithm
    coverage = []
    current_coverage = 0
    for pos, change in events:
        current_coverage += change
        coverage.append({'chrom': chrom, 'start': pos, 'end': pos, 'value': current_coverage})
    return pd.DataFrame(coverage)

# Read data
calls = pd.read_csv('~/projects/mCAs_WGS/RAP/calls/WGS_500k.calls.txt', sep='\t')
genomic_data = pd.read_csv('~/projects/mCAs_WGS/RAP/cisGWAS/results/CN-LOH.GWAS.FDR.01.txt', sep='\t')
df_phase = pd.read_csv('~/projects/mCAs_WGS/RAP/cisGWAS/results/refined_phase/FDR.01.burden.allelic_shift.GWAS.txt.gz', sep='\t')

# Prepare genomic_data
df_phase['gene'] = df_phase['mask'].str.replace(r"\..*", "", regex=True)
genomic_data = pd.merge(genomic_data, df_phase, left_on='gene', right_on='gene')
genomic_data['start'] = genomic_data['pos'] * 1e6
genomic_data['end'] = genomic_data['start']

# Prepare coverage data
df_list = []
for chrom in [f"chr{i}" for i in range(1, 23)]:
    df_chrom = calls[(calls['chr'] == chrom) & (calls['type'] == "CN-LOH")]
    df_list.append(count_coverage_df(df_chrom, chrom))
coverage_data = pd.concat(df_list, ignore_index=True)


genomic_data['two_sided_tail_prob'] = 2*np.minimum(
    scipy.stats.beta(1/2+genomic_data['carrier_overrep'], 1/2+genomic_data['carrier_underrep']).cdf(0.5),
    scipy.stats.beta(1/2+genomic_data['carrier_overrep'], 1/2+genomic_data['carrier_underrep']).sf(0.5)
)
gray_genes = set(genomic_data[(genomic_data['two_sided_tail_prob'] > 0.1)]['gene'])
purple_genes = set(genomic_data[genomic_data['carrier_overrep'] > genomic_data['carrier_underrep']]['gene']).difference(gray_genes)
green_genes =  set(genomic_data[genomic_data['carrier_overrep'] < genomic_data['carrier_underrep']]['gene']).difference(gray_genes)

In [ ]:
from pycirclize import Circos

# Load hg38 dataset (https://github.com/moshi4/pycirclize-data/tree/main/eukaryote/hg38)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
chr_bed_df = pd.read_csv(chr_bed_file, sep='\t')
chr_bed_df[~chr_bed_df.iloc[:,0].isin(['chrX', 'chrY'])].to_csv(chr_bed_file, sep='\t', index=False)

# Initialize Circos from BED chromosomes
circos = Circos.initialize_from_bed(chr_bed_file, space=2)

# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((86, 90), cytoband_file)


theta_adjust = defaultdict(int)
theta_adjust['ERG'] = -5
theta_adjust['LZTR1'] = -7
theta_adjust['CHEK2'] = -4.5
theta_adjust['PRR14L'] = -2
theta_adjust['IL2RB'] = 0

theta_adjust['TNFRSF8'] = -1.5
theta_adjust['RCC2'] = 0.5
theta_adjust['MPL'] = -0
theta_adjust['DMAP1'] = 2.5
theta_adjust['TM2D1'] = 3.5
theta_adjust['JAK1'] = 6
theta_adjust['COQ8A'] = -1
theta_adjust['FH'] = 1

theta_adjust['IRF4'] = -2
theta_adjust['HUS1B'] = 2

theta_adjust['NOTCH1'] = 2

theta_adjust['ATM'] = 1.5
theta_adjust['CBL'] = 3

theta_adjust['STUB1'] = 1

# Plot chromosome name

fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(111, polar=True)

for sector in circos.sectors:
    rotation = -(sector.rad_lim[0] + sector.rad_lim[1]) / 2 * 180 / np.pi
    sector.text(
        sector.name[3:], # Remove 'chr' prefix
        r = 80,
        size= 18, 
        rotation=rotation, 
        adjust_rotation=False
    )
    for idx, row in genomic_data.query(f"chr == '{sector.name}'").iterrows():
        
        pos = int(row['start'])
        
        label = row['gene']
        color = '#c86bdf' if label in purple_genes else '#4caf50' if label in green_genes else 'k'
        
        theta = ((sector.rad_lim[1] - sector.rad_lim[0]) * pos / (sector.size ) + sector.rad_lim[0]) * 180 / np.pi
        gene_theta = theta
        theta += theta_adjust[label]
        rotation = -theta-90

        
        ax.text(
            np.radians(theta),
            100,
            rf'$\it{{{label}}}$', 
            color=color, 
            size=16, 
            ha="right" if theta > 180 else "left", 
            va="center", 
            rotation=rotation if theta > 180 else rotation + 180,
            rotation_mode='anchor'
        )
        ax.plot(
            np.radians([theta, gene_theta]),
            [99, 91],
            color='k',
            linewidth=1
        )

    cov = coverage_data.query(f"chrom == '{sector.name}'")
    angles = (cov['start'] / sector.size) * (sector.rad_lim[1] - sector.rad_lim[0]) + sector.rad_lim[0]
    radii = 75-cov['value']/30
    ax.plot(angles, radii, color='k', linewidth=1)
    ax.fill_between(angles, radii, 75, color='gold', alpha=1)
    ax.plot(angles, [75]*len(angles), color='k', linewidth=1)

fig = circos.plotfig(ax=ax)
plt.savefig('GWAS_plots/GWAS_summary.pdf', bbox_inches='tight', transparent=True)